# 01 · Exploratory data analysis of the four NetFlow v3 datasets

Run `python scripts/prepare_data.py` first. This notebook reads the subsampled Parquet files and shows (1) the class balance, (2) the schema, and (3) how far each feature's distribution shifts between datasets, which is the reason cross-dataset transfer is hard.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp

from src.utils import load_config, set_seed, resolve
from src.features import feature_sets, check_schema

cfg = load_config()
set_seed(cfg['seed'])
DATA_DIR = resolve(cfg['paths']['processed_dir'])   # change to a toy folder for a dry run
FIG_DIR = resolve(cfg['paths']['results_dir']) / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

dfs = {k: pd.read_parquet(DATA_DIR / f'{k}.parquet')
       for k in cfg['datasets'] if (DATA_DIR / f'{k}.parquet').exists()}
{k: v.shape for k, v in dfs.items()}

## 1. Schema check
Confirms the 43 v2 features are present and lists the v3-only columns found in each file.

In [ ]:
pd.DataFrame([check_schema(df, k) for k, df in dfs.items()])

## 2. Class balance

In [ ]:
rows = []
for k, df in dfs.items():
    vc = df['Attack'].value_counts(normalize=True).mul(100).round(2)
    rows.append(vc.rename(k))
balance = pd.concat(rows, axis=1).fillna(0)
balance

In [ ]:
attack_rate = pd.Series({k: 100 * df['Label'].mean() for k, df in dfs.items()})
ax = attack_rate.sort_values().plot.barh(color='#4C72B0')
ax.set_xlabel('Attack flows (%)'); ax.set_title('Attack rate per dataset (subsample)')
plt.tight_layout(); plt.savefig(FIG_DIR / 'attack_rate.png', dpi=200); plt.show()

## 3. Distribution shift between datasets
For every feature and every pair of datasets we compute the two-sample Kolmogorov–Smirnov statistic on benign flows (0 = identical, 1 = no overlap). High values mark features a model may learn as dataset-specific shortcuts.

In [ ]:
keys = list(dfs)
feats = feature_sets(next(iter(dfs.values())).columns)['v3']
feats = [f for f in feats if all(f in df.columns for df in dfs.values())]
rows = []
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        a = dfs[keys[i]].query('Label == 0'); b = dfs[keys[j]].query('Label == 0')
        if len(a) < 2 or len(b) < 2:
            continue
        for f in feats:
            rows.append({'pair': f'{keys[i]}-{keys[j]}', 'feature': f,
                         'ks': ks_2samp(a[f], b[f]).statistic})
ks = pd.DataFrame(rows).pivot(index='feature', columns='pair', values='ks')
ks['mean'] = ks.mean(axis=1)
ks = ks.sort_values('mean', ascending=False)
ks.to_csv(resolve(cfg['paths']['results_dir']) / 'ks_shift_benign.csv')
ks.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))
im = ax.imshow(ks.drop(columns='mean').values, aspect='auto', cmap='viridis', vmin=0, vmax=1)
ax.set_yticks(range(len(ks))); ax.set_yticklabels(ks.index, fontsize=7)
ax.set_xticks(range(ks.shape[1] - 1)); ax.set_xticklabels(ks.columns[:-1], rotation=45, ha='right')
fig.colorbar(im, label='KS statistic (benign flows)')
ax.set_title('Per-feature distribution shift between datasets')
plt.tight_layout(); plt.savefig(FIG_DIR / 'ks_shift_heatmap.png', dpi=200); plt.show()

## Notes for the paper
- Report the subsample sizes and class balance table (Section 3, Data).
- Features with mean KS close to 1 are candidates for the shortcut-feature analysis in week 6.